# 2. Original H1 experiment: T4 / FP16 amendment
Attach the stage1 code bundle and `stage1-online-inputs.zip`. Select T4 x2; this runner uses GPU 0 only.
The frozen primary matrix remains 3 seeds, RMS + Taper-minus, both continuation branches.
Read `docs/scientific-runbook.md`. Modes: `preflight`, then a NEW SESSION `start`, then `resume` per saved chunk.
No automatic primary launch: set RUN_PRIMARY explicitly after preflight passes.

Budget amendment 2026-09-30: up to 50 cumulative free GPU hours across sessions/weeks. Fixed data, seeds, token budgets and scientific thresholds are unchanged. Weekly quota resets never reset EXTERNAL_GPU_HOURS or the experiment ledger. Use a fresh preflight with this code/config; older preflight evidence remains historical.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, zipfile, shutil, tempfile, time
inputs = Path('/kaggle/input')
roots = sorted({p.parent.parent.parent for p in inputs.rglob('pilot.py')
                if p.parent.name == 'domain_shift_forgetting' and p.parent.parent.name == 'src'})
if roots:
    assert len(roots) == 1, f'Attach exactly one stage1 code bundle: {roots}'
    code = roots[0]
else:
    bundles = list(inputs.rglob('domain-shift-stage1.zip'))
    assert len(bundles) == 1, 'Attach domain-shift-stage1.zip as a private dataset'
    code = Path(tempfile.mkdtemp(prefix='stage1-code-'))
    with zipfile.ZipFile(bundles[0]) as z:
        for name in z.namelist():
            assert (code / name).resolve().is_relative_to(code.resolve())
        z.extractall(code)
env = os.environ | {'PYTHONPATH': str(code / 'src'), 'CUBLAS_WORKSPACE_CONFIG': ':4096:8'}
def cli(module, *args):
    subprocess.run([sys.executable, '-m', 'domain_shift_forgetting.' + module,
                    *map(str, args)], env=env, check=True)
print('Code:', code)


In [ ]:
env['CUDA_VISIBLE_DEVICES'] = '0'
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pytest', 'datasketch'], check=True)
# Set these from the CURRENT live-session timer and weekly quota, not their nominal limits.
MODE = 'preflight'  # later: 'start' or 'resume'
SESSION_REMAINING_MINUTES = None
WEEKLY_GPU_HOURS_REMAINING = None
# Cumulative stage GPU time OUTSIDE the runner ledger: old smoke tests, idle/setup,
# preflight(s), external-restore checks and exports. Never reduce this on resume.
EXTERNAL_GPU_HOURS = None
RUN_PRIMARY = False
RECOVER_UNCLEAN = False  # True only after confirming the previous worker ended.
assert MODE in ('preflight', 'start', 'resume')
assert all(isinstance(x, (int, float)) and x >= 0 for x in
           (SESSION_REMAINING_MINUTES, WEEKLY_GPU_HOURS_REMAINING, EXTERNAL_GPU_HOURS)), 'Fill the three usage values first.'
remaining_at_entry = min(SESSION_REMAINING_MINUTES, WEEKLY_GPU_HOURS_REMAINING * 60)
settings_time = time.monotonic()
def remaining():
    value = remaining_at_entry - (time.monotonic() - settings_time) / 60
    assert value > 10, 'Save outputs and start a fresh session; less than 10 minutes remain.'
    return value
scratch = Path(tempfile.mkdtemp(prefix='stage1-'))  # scratch is not the durable backup
policy = code / 'configs/kaggle-stage1.json'
# Kaggle commonly unpacks uploaded archives. Handle a retained ZIP as well.
online_roots = [p.parent for p in inputs.rglob('manifest.json')
                if p.parent.name == 'online' and json.loads(p.read_text()).get('schema') == 'pilot-arrays-v1']
if not online_roots:
    bundles = list(inputs.rglob('stage1-online-inputs.zip'))
    assert len(bundles) == 1, 'Attach the original prepared online-inputs ZIP.'
    target = scratch / 'inputs'
    with zipfile.ZipFile(bundles[0]) as z:
        for name in z.namelist():
            assert (target / name).resolve().is_relative_to(target.resolve())
        z.extractall(target)
    online_roots = [target / 'online']
assert len(online_roots) == 1, f'Ambiguous original corpus: {online_roots}'
data = online_roots[0]
orders, upstream = data.parent / 'orders', data.parent / 'upstream'
root = scratch / 'run'
print('Mode:', MODE, 'original data:', data, 'working run:', root)


In [ ]:
# PRECHECK PHASE: real-array correctness, genuine calibration, phase timings, fixed-event projection.
if MODE == 'preflight':
    preflight = scratch / 'preflight'
    try:
        cli('pilot', 'preflight', '--data-dir', data, '--orders', orders,
            '--config', policy, '--output', preflight, '--tests', code / 'tests',
            '--prior-gpu-hours', EXTERNAL_GPU_HOURS, '--session-remaining-minutes', remaining())
    finally:
        stamp = str(int(time.time()))
        if preflight.exists():
            cli('pilot', 'export', '--root', preflight, '--output', f'/kaggle/working/stage1-preflight-{stamp}.zip')
            cli('pilot', 'export', '--root', preflight, '--reports-only',
                '--output', f'/kaggle/working/stage1-preflight-reports-{stamp}.zip')
    benchmark = json.loads((preflight / 'benchmark.json').read_text())
    if benchmark.get('status') == 'passed':
        print('Persist the full preflight archive outside this session. Reattach it in a NEW SESSION with MODE=start for the required restore check.')
    else:
        print('STOP: preflight is infeasible under the frozen budget. Save both archives, stop the GPU session, and review the report. Do not use MODE=start.')


In [ ]:
# START: prove recovery from a persisted preflight checkpoint before admitting seed 101.
# RESUME: attach the LAST complete stage1-resume ZIP from the previous chunk.
if MODE == 'start':
    assert RUN_PRIMARY, 'Review preflight; set RUN_PRIMARY=True when ready to begin the admitted matrix.'
    proofs = list(inputs.rglob('resume-proof.json'))
    if proofs:
        assert len(proofs) == 1, 'Attach exactly one preflight archive.'
        preflight = proofs[0].parent
    else:
        archives = [p for p in inputs.rglob('stage1-preflight-*.zip') if 'reports' not in p.name]
        assert len(archives) == 1, 'Attach the persisted preflight ZIP, not reports-only.'
        preflight = scratch / 'persisted-preflight'
        cli('pilot', 'import', '--archive', archives[0], '--output', preflight)
    admission = json.loads((preflight / 'benchmark.json').read_text())
    assert admission.get('status') == 'passed', 'Preflight was infeasible: retain reports and review performance/budget before starting.'
    proof_result = scratch / 'external-resume.json'
    cli('pilot', 'verify-resume', '--preflight', preflight, '--data-dir', data,
        '--orders', orders, '--output', proof_result)
    minimum_external = json.loads((preflight / 'benchmark.json').read_text())['prior_gpu_hours']
    assert EXTERNAL_GPU_HOURS >= minimum_external, f'Include prior validation/benchmark time: at least {minimum_external:.4f} hours, plus subsequent setup/idle.'
    measured_budget = json.loads((preflight / 'benchmark.json').read_text())
    updated_projection = measured_budget['projection_with_margin_hours'] + 1.15 * (EXTERNAL_GPU_HOURS - minimum_external)
    assert updated_projection <= json.loads(policy.read_text())['active_gpu_hours_cap'], 'Updated cumulative usage exceeds the documented experiment budget.'
    cli('pilot', 'freeze', '--data-dir', data, '--orders', orders, '--config', policy,
        '--output', root, '--preflight', preflight, '--sources', upstream, '--resume-validation', proof_result)
elif MODE == 'resume':
    assert RUN_PRIMARY, 'Set RUN_PRIMARY=True to resume the SAME frozen matrix.'
    archives = [p for p in inputs.rglob('stage1-resume-*.zip') if 'reports' not in p.name]
    if archives:
        assert len(archives) == 1, 'Attach only the LAST complete resume archive.'
        cli('pilot', 'import', '--archive', archives[0], '--output', root)
    else:
        receipts = [p for p in inputs.rglob('archive-receipt.json') if (p.parent / 'freeze.json').exists()]
        assert len(receipts) == 1, 'Attach the last complete resume ZIP (Kaggle may extract it).'
        cli('pilot', 'import-directory', '--source', receipts[0].parent, '--output', root)


In [ ]:
if MODE in ('start', 'resume'):
    try:
        cli('pilot', 'run', '--root', root, '--data-dir', data, '--orders', orders,
            '--session-remaining-minutes', remaining(), '--external-gpu-hours', EXTERNAL_GPU_HOURS,
            *(['--recover-unclean'] if RECOVER_UNCLEAN else []))
    finally:
        stamp = str(int(time.time()))
        if (root / 'freeze.json').exists():
            try:
                cli('pilot', 'report', '--root', root)
            finally:
                cli('pilot', 'export', '--root', root, '--output', f'/kaggle/working/stage1-resume-{stamp}.zip')
            cli('pilot', 'export', '--root', root, '--reports-only',
                '--output', f'/kaggle/working/stage1-reports-{stamp}.zip')
    print((root / 'status.json').read_text())
    print((root / 'decision-report.txt').read_text())
    print('Persist ALL printed archives before stopping. Send the small stage1-reports ZIP for review.')


A pause is normal: a chunk deadline, storage reserve or a newly trained prefix requires a durable export.
Keep every full archive; later archives reference earlier weight snapshots. Never resume from the reports-only ZIP.
An incomplete matrix is not a negative scientific result. Do not alter thresholds, seeds, endpoints or budgets after inspecting results.
